# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code or Rust code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            This lab uses FREE models only: Ollama (local, no API key needed) and Groq (free tier API). Install Ollama from https://ollama.com or get a free Groq API key from https://console.groq.com
            </span>
        </td>
    </tr>
</table>

In [46]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [47]:
# Using FREE models only: Groq (free tier) and Ollama (local, no API key needed)

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"✓ Groq API Key found (begins {groq_api_key[:4]})")
    print("  Using Groq's FREE tier API")
else:
    print("✗ Groq API Key not set")
    print("  Get a free key at: https://console.groq.com")

# Check if Ollama is available
print("\nChecking Ollama (local, free):")
try:
    result = subprocess.run(['ollama', 'list'], capture_output=True, text=True, timeout=5)
    print("✓ Ollama is installed and available")
    print("\nAvailable models:")
    print(result.stdout)
except FileNotFoundError:
    print("✗ Ollama not found. Please install from https://ollama.com")
except Exception as e:
    print(f"Error checking Ollama: {e}")



✓ Groq API Key found (begins gsk_)
  Using Groq's FREE tier API

Checking Ollama (local, free):
✓ Ollama is installed and available

Available models:
NAME                ID              SIZE      MODIFIED    
deepseek-r1:1.5b    e0979632db5a    1.1 GB    11 days ago    
llama3.2:latest     a80c4f17acd5    2.0 GB    2 weeks ago    
gemma3:1b           8648f39daa8f    815 MB    2 weeks ago    



In [48]:
# Connect to FREE services only

# Groq (free tier API)
groq_url = "https://api.groq.com/openai/v1"
if groq_api_key:
    groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
    print("✓ Connected to Groq API (free tier)")
else:
    groq = None
    print("✗ Groq not available (no API key)")

# Ollama (local, free)
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
print("✓ Connected to Ollama at", ollama_url)



✓ Connected to Groq API (free tier)
✓ Connected to Ollama at http://localhost:11434/v1


In [49]:
# FREE models only!

# Groq models (free tier API - very fast!)
GROQ_LLAMA_70B = "llama-3.3-70b-versatile"
GROQ_LLAMA_8B = "llama-3.1-8b-instant"

# Ollama models (local, free - check what you have installed)
OLLAMA_MODELS = [
    "llama3.2:latest",
    "deepseek-r1:1.5b",
    "gemma3:1b",
]

# Build models list and clients dict
models = []
clients = {}

if groq:
    models.extend([GROQ_LLAMA_70B, GROQ_LLAMA_8B])
    clients[GROQ_LLAMA_70B] = groq
    clients[GROQ_LLAMA_8B] = groq

models.extend(OLLAMA_MODELS)
for model in OLLAMA_MODELS:
    clients[model] = ollama

print(f"\nAvailable FREE models ({len(models)} total):")
for model in models:
    print(f"  - {model}")


Available FREE models (5 total):
  - llama-3.3-70b-versatile
  - llama-3.1-8b-instant
  - llama3.2:latest
  - deepseek-r1:1.5b
  - gemma3:1b


In [50]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': True,
 'rustc': {'path': 'C:\\Users\\faisal.khan\\.cargo\\bin\\rustc.EXE',
  'version': 'rustc 1.93.0 (254b59607 2026-01-19)',
  'host_triple': 'x86_64-pc-windows-msvc',
  'release': '1.93.0',
  'commit_hash': '254b59607d4417e9dffbc307138ae5c86280fe4c'},
 'cargo': {'path': 'C:\\Users\\faisal.khan\\.cargo\\bin\\cargo.EXE',
  'version': 'cargo 1.93.0 (083ac5135 2025-12-15)'},
 'rustup': {'path': 'C:\\Users\\faisal.khan\\.cargo\\bin\\rustup.EXE',
  'version': 'rustup 1.28.2 (e4f3ad6f8 2025-04-28)',
  'active_toolchain': 'stable-x86_64-pc-windows-msvc (default)',
  'default_toolchain': '',
  'toolchains': ['stable-x86_64-pc-windows-msvc (active, default)'],
  'targets_installed': ['x86_64-pc-windows-msvc']},
 'rust_analyzer': {'path': 'C:\\Users\\faisal.khan\\.cargo\\bin\\rust-analyzer.EXE'},
 'env': {'CARGO_HOME': 'C:\\Users\\faisal.khan\\.cargo',
  'RUSTUP_HOME': 'C:\\Users\\faisal.khan\\.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': ['"C:\

In [51]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

# Use the first available model's client
client = clients[models[0]]
response = client.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

You already have the Rust toolchain installed on your system, so you don't need to install anything else.

To compile and run the `main.rs` file with the fastest possible runtime performance, you can use the following commands:

*   `compile_command`: 
    ```markdown
"C:\\Users\\faisal.khan\\.cargo\\bin\\rustc.EXE" main.rs -o main.exe -C opt-level=3 -C target-cpu=native
```
*   `run_command`: 
    ```markdown
main.exe
```

These commands tell the Rust compiler (`rustc`) to compile the `main.rs` file with the following options:
*   `-C opt-level=3` : Optimize the code for the fastest possible runtime performance. This can increase compilation time.
*   `-C target-cpu=native` : Optimize the code for your specific CPU. This can further improve runtime performance.
*   `-o main.exe` : Specify the output file name as `main.exe`.

After compiling, the `main.exe` file is executed directly.

You can use these commands in your Python script as follows:

```python
import subprocess

compile_command = ["C:\\Users\\faisal.khan\\.cargo\\bin\\rustc.EXE", "main.rs", "-o", "main.exe", "-C", "opt-level=3", "-C", "target-cpu=native"]
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

run_command = ["main.exe"]
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

return run_result.stdout
```

## For C++, overwrite this with the commands from yesterday, or for Rust, use the new commands

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [52]:
compile_command = [
    "/Users/ed/.cargo/bin/rustc",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "codegen-units=1",
    "-C", "lto=fat",
    "-C", "panic=abort",
    "-C", "strip=symbols",
    "-o", "main",
]

run_command = ["./main"]


## And now, on with the main task

In [53]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [54]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [55]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [56]:
def port(model, python):
    client = clients[model]
    response = client.chat.completions.create(model=model, messages=messages_for(python))
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [57]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [58]:
# Use the commands from GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [59]:
python_hard = """import time

# Simple computation with timing
start_time = time.time()

# Do some simple work
result = 0
for i in range(1000000):
    result = result + i

end_time = time.time()

print("Result:", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [60]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Your Experiment Results

Test the FREE models and record your Rust execution times:

**Groq Models:**
- Llama 3.3 70B: 
  - Result: ___ (Success/Fail)
  - Rust Time: ___ seconds
- Llama 3.1 8B: 
  - Result: ___ (Success/Fail)
  - Rust Time: ___ seconds

**Ollama Local Models:**
- llama3.2:latest: 
  - Result: ___ (Success/Fail)
  - Rust Time: ___ seconds
- deepseek-r1:1.5b: 
  - Result: ___ (Success/Fail)
  - Rust Time: ___ seconds
- gemma3:1b: 
  - Result: ___ (Success/Fail)
  - Rust Time: ___ seconds

**Note:** This is a challenging task - converting Python to Rust is more complex than Python to C++. Don't worry if some models fail!

## About Python-to-Rust Conversion with FREE Models

This is a **challenging exercise** that tests the limits of free models:

**Why Rust?**
- Extremely fast execution (often faster than C++)
- Memory safe without garbage collection
- Complex syntax that's harder for models to generate

**Using FREE Models:**
- **Groq Free Tier:** Fast inference, good for complex tasks
- **Ollama Local:** Smaller models may struggle with Rust syntax

**What to Expect:**
- Some models may fail to generate valid Rust code
- Successful conversions will show impressive speedups
- This demonstrates how challenging Python-to-Rust is compared to Python-to-C++

**Tips:**
- Start with simpler Python code if you get errors
- Check the Rust output for compilation errors
- Install larger Ollama models for better results: `ollama pull qwen2.5-coder:7b`

In [61]:
# Calculate your speedup!
# If you get successful Rust compilation, compare Python vs Rust times
# Example: If Python takes 0.067s and Rust takes 0.0003s, that's 223x faster!
print("Test the models above and record your results!")

Test the models above and record your results!


In [62]:
compile_and_run(cpp.value)

FileNotFoundError: [WinError 2] The system cannot find the file specified

Traceback (most recent call last):
  File "d:\projects\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\projects\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\projects\llm_engineering\.venv\Lib\site-packages\anyio\to_thread.py", line 56, in run_sync
    return await get